In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
BASE_DIR = '/content/drive/MyDrive/Глубокое обучение/data/'

TEST_PATH = os.path.join(BASE_DIR, 'test.csv')
SCALER_PATH = os.path.join(BASE_DIR, 'scaler.pkl')
LR_MODEL_PATH = os.path.join(BASE_DIR, 'lr_model.pkl')
RF_MODEL_PATH = os.path.join(BASE_DIR, 'rf_model.pkl')
NN_MODEL_PATH = os.path.join(BASE_DIR, 'best_nn_model.pth')
OUTPUT_PATH = os.path.join(BASE_DIR, 'sample_submission.csv')

scaler = joblib.load(SCALER_PATH)
lr_model = joblib.load(LR_MODEL_PATH)
rf_model = joblib.load(RF_MODEL_PATH)

class CVDNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
    def forward(self, x): return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#scaler хранит количество признаков, на которых обучался
input_dim = scaler.n_features_in_ if hasattr(scaler, 'n_features_in_') else 13
nn_model = CVDNet(input_dim=input_dim).to(device)
nn_model.load_state_dict(torch.load(NN_MODEL_PATH, map_location=device, weights_only=True))
nn_model.eval()

test_df = pd.read_csv(TEST_PATH)
X_test_raw = test_df.drop('ID', axis=1)

#Масштабируем
X_test_scaled = scaler.transform(X_test_raw)

#Логистическая регрессия
lr_preds = lr_model.predict(X_test_scaled)

#Случайный лес
rf_preds = rf_model.predict(X_test_scaled)

#Нейронная сеть (PyTorch)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
with torch.no_grad():
    nn_logits = nn_model(X_test_t)
    nn_probs = torch.sigmoid(nn_logits).cpu().numpy().flatten()
nn_preds = (nn_probs >= 0.5).astype(int)

#Таблица с предсказаниями
final_submission = pd.DataFrame({
    'ID': test_df['ID'],
    'LR_Prediction': lr_preds,
    'RF_Prediction': rf_preds,
    'NN_Prediction': nn_preds,
})

#Модальное предсказание финального класса
final_submission['class'] = final_submission[['LR_Prediction', 'RF_Prediction', 'NN_Prediction']].mode(axis=1)[0].astype(int)
#Удаление предварительных столбцов
final_submission = final_submission.drop(columns=['LR_Prediction', 'RF_Prediction', 'NN_Prediction'])
#Сохранение датафрейма
final_submission.to_csv('./drive/MyDrive/Глубокое обучение/final_submission.csv', index=False)

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
